JAI SHREE RAM

Same as `FINAL SURAKSHANET.ipynb` + MACD Devanagari→Hinglish (Roman) for WhatsApp support.


In [ ]:
%uv pip install -q transformers datasets accelerate evaluate scikit-learn sentencepiece indic-transliteration


In [ ]:
import transformers, datasets, torch
print(transformers.__version__, datasets.__version__, torch.__version__)
torch.cuda.is_available()

In [ ]:
torch.cuda.get_device_name(0)

In [ ]:
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
# just a simple pretrained language model

In [ ]:
SEED = 42
MAX_LENGTH = 400
ID2LABEL = {0: "abusive", 1: "non-abusive"}
LABEL2ID = {"abusive": 0, "non-abusive": 1}
OUTPUT_DIR = "/vol/checkpoints/pt"

In [ ]:
from datasets import load_dataset
macd_train = load_dataset("csv",data_files="https://raw.githubusercontent.com/ShareChatAI/MACD/main/dataset/hindi_train.csv",)["train"]

In [ ]:
macd_val = load_dataset("csv",data_files="https://raw.githubusercontent.com/ShareChatAI/MACD/main/dataset/hindi_val.csv",)["train"]

In [ ]:
macd_test = load_dataset("csv",data_files="https://raw.githubusercontent.com/ShareChatAI/MACD/main/dataset/hindi_test.csv",)["train"]

## Hinglish (minimal add)
Romanize MACD with same labels. Cached under `/vol/checkpoints/hinglish_cache`.


In [ ]:
from pathlib import Path
import re
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

HINGLISH_CACHE = Path("/vol/checkpoints/hinglish_cache")
HINGLISH_CACHE.mkdir(parents=True, exist_ok=True)
DEVANAGARI = re.compile(r"[\u0900-\u097F]+")
_diacritic = re.compile(r"[\u0300-\u036f]")

def to_hinglish(text: str) -> str:
    if not text:
        return text
    # Only romanize Devanagari spans; keep emoji/English/hashtags
    def repl(m):
        roman = transliterate(m.group(0), sanscript.DEVANAGARI, sanscript.ITRANS)
        return _diacritic.sub("", roman)
    return DEVANAGARI.sub(repl, text)

def add_hinglish(example):
    return {"text": to_hinglish(example["text"]), "label": example["label"]}

def load_or_build_hinglish(name, ds):
    cache = HINGLISH_CACHE / f"{name}.csv"
    if cache.exists() and cache.stat().st_size > 0:
        print("cache hit", cache)
        return load_dataset("csv", data_files=str(cache))["train"]
    print("transliterating", name, "n=", len(ds))
    out = ds.map(add_hinglish)
    out.to_csv(str(cache), index=False)
    return out

macd_train_hinglish = load_or_build_hinglish("train", macd_train)
macd_val_hinglish = load_or_build_hinglish("val", macd_val)
macd_test_hinglish = load_or_build_hinglish("test", macd_test)
print(macd_train[0])
print(macd_train_hinglish[0])


In [ ]:
macd_train

In [ ]:
davidson = load_dataset("csv",data_files="https://raw.githubusercontent.com/t-davidson/hate-speech-and-offensive-language/master/data/labeled_data.csv")["train"]

In [ ]:
def convert_labels(example):
    if example["class"] in [0,1]:
        return {"label": 0}
    else:
        return {"label" : 1}

In [ ]:
davidson

In [ ]:
davidson = davidson.map(convert_labels)

In [ ]:
davidson = davidson.rename_column("tweet","text")

In [ ]:
rem = [x for x in davidson.column_names if x not in ["text","label"]]
davidson = davidson.remove_columns(rem)


davidson = davidson.train_test_split(
    test_size=0.10,
    seed=SEED,
)


In [ ]:
davidson

In [ ]:
davidson_train = davidson["train"]
davidson_holdout = davidson["test"]

In [ ]:
macd_train.features.type

In [ ]:
davidson_train.features.type

In [ ]:
from datasets import concatenate_datasets
dataset = concatenate_datasets([davidson_train, macd_train, macd_train_hinglish])


In [ ]:
dataset=dataset.shuffle(seed=SEED)

In [ ]:
print(dataset.column_names)
print(dataset[10])

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
check = tokenizer("यह एक परीक्षण संदेश है", padding=True, truncation=True, return_tensors="pt")


In [ ]:
tokenizer.decode(check["input_ids"][0])

In [ ]:
def tokenize_function(example):
    return tokenizer(example["text"],  truncation=True)

In [ ]:
dataset = dataset.map(tokenize_function,batched=True)

In [ ]:
tokenized_train = dataset

In [ ]:
macd_val_both = concatenate_datasets([macd_val, macd_val_hinglish]).shuffle(seed=SEED)
tokenized_val = macd_val_both.map(tokenize_function, batched=True)


In [ ]:
tokenized_train

In [ ]:
tokenized_val

In [ ]:
tokenized_val[0]

In [ ]:
features = [
    {
        key : value 
        for key,value in tokenized_train[index].items()
        if key!="text"
    }
    for index in range(4)
]

In [ ]:
features


In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

batch_smoke_test = data_collator(features)
batch_smoke_test

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2,id2label=ID2LABEL,
    label2id=LABEL2ID,
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 and are newly initialized: ['classifier.bias', 'classifier.weight']


You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


this line actually means that uska jo head hai that is changed to somewhat which is expected by AutoModelForSequenceClassification as per https://huggingface.co/learn/llm-course/chapter2/2


In [ ]:
print(model.config.label2id)


classifier's weights and biases are randomly initialized so it expects the model to be retrained on some actual data 

In [ ]:
smoke_outputs = model(**batch_smoke_test)

just did a simple forward pass to see the output , this did not include any optimization or back prop


In [ ]:
smoke_outputs.loss

In [ ]:
smoke_outputs.logits

trainer.train hi actual training karega 

In [ ]:
tokenizer.model_max_length

In [ ]:
model.config.max_position_embeddings

In [ ]:
from transformers import TrainingArguments


training_args = TrainingArguments("test-trainer", 
                                  eval_strategy="epoch",
                                  save_strategy="epoch",
                                  load_best_model_at_end=True,
                                  metric_for_best_model="f1_macro",
                                  greater_is_better=True,
                                  report_to="none"
                                 )
    


In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

def compute_metrics(eval_predictions):
    logits, labels = eval_predictions
    predictions = np.argmax(logits,axis=-1)

    return {
        "accuracy" : accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels,predictions,average="macro"),
        "precision_macro": precision_score(labels, predictions, average="macro",zero_division=0),
        "recall_macro": recall_score(labels, predictions, average="macro",zero_division=0),
    }


note : compute metrics are not needed for updation of weights because that part is actually done by the loss and the optimizer , the compute metrics is actually for finally saving the model which we feel has done the best over the differnt checkpoints (i mean the epochs in our case)

note that the labels in this is the ground truth value and logits are our 

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

In [ ]:
trainer.train()

internally traning does this:
batch bnao -> forward pass -> calculate loss -> backward pass -> optimizer step -> weights update ->epoch ke end par validation par run -> compute metrics -> checkpoint save

In [ ]:
trainer.evaluate()


In [ ]:
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved to:", OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))

In [ ]:
tokenized_macd_test = macd_test.map(
    tokenize_function,
    batched=True,
)
tokenized_macd_test_hinglish = macd_test_hinglish.map(
    tokenize_function,
    batched=True,
)
tokenized_davidson_holdout = davidson_holdout.map(
    tokenize_function,
    batched=True,
)


In [ ]:
macd_test_output = trainer.predict(
    tokenized_macd_test
)
print("MACD Devanagari test:", macd_test_output.metrics)

macd_test_hinglish_output = trainer.predict(
    tokenized_macd_test_hinglish
)
print("MACD Hinglish test:", macd_test_hinglish_output.metrics)


In [ ]:
davidson_test_output = trainer.predict(
    tokenized_davidson_holdout
)
print(davidson_test_output.metrics)

shrink the fine-tuned MiniLM for on device chrome and infernce with ONXXweb runtime !
### Goal :  run inference with ONXX web runtime

### STEPS 
1. measure curr size
2. get ONXX FP32 export with https://huggingface.co/docs/optimum/quicktour#onnx-runtime
3. now quantize this INT8 model
4. measure size and accuracy delta
   

In [ ]:
from pathlib import Path

def print_model_size(model_dir, title=None):
    """Print per-file and total on-disk size (MB). Follows symlinks."""
    root = Path(model_dir).resolve()
    if title:
        print(f"=== {title} ===")
    print("resolved:", root)
    assert root.is_dir(), f"missing directory: {root}"

    files = sorted(p for p in root.rglob("*") if p.is_file())
    total = 0
    for p in files:
        mb = p.stat().st_size / 1024**2
        total += mb
        print(f"  {str(p.relative_to(root)):40s} {mb:8.2f} MB")
    print(f"TOTAL: {total:.2f} MB\n")
    return total

print_model_size(OUTPUT_DIR, title="Checkpoint Directory")


In [ ]:
%uv pip install -q "optimum[onnxruntime]" onnx onnxruntime

In [ ]:
import optimum
print("optimum ready")

In [ ]:
from pathlib import Path

ONNX_FP32_DIR= "/vol/checkpoints/onnx_fp32"
Path(ONNX_FP32_DIR).mkdir(parents=True, exist_ok=True)

ONNX_INT8_DIR = "/vol/checkpoints/onnx_int8"
Path(ONNX_INT8_DIR).mkdir(parents=True, exist_ok=True)


In [ ]:
from pathlib import Path
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

# this time we are loading ths model directly 
model_checkpoint = OUTPUT_DIR
save_directory = ONNX_FP32_DIR


# Load a model from transformers and export it to ONNX
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
ort_model = ORTModelForSequenceClassification.from_pretrained(model_checkpoint, export=True)

# Save the ONNX model and tokenizer
ort_model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

In [ ]:
print_model_size(ONNX_FP32_DIR, "ONNX FP32")

dynamic int8 quantization :-
> AutoQuantizationConfig + ORTQuantizer -> quanitize(...)

In [ ]:
from optimum.onnxruntime.configuration import AutoQuantizationConfig
from optimum.onnxruntime import ORTQuantizer
from pathlib import Path
from transformers import AutoTokenizer


# Define the quantization methodology
qconfig = AutoQuantizationConfig.avx2(is_static=False, per_channel=False)
quantizer = ORTQuantizer.from_pretrained(ort_model)

save_directory = ONNX_INT8_DIR


# Apply dynamic quantization on the model
quantizer.quantize(save_dir=save_directory, quantization_config=qconfig)

# same tokenizer 
tokenizer = AutoTokenizer.from_pretrained(ONNX_FP32_DIR)
tokenizer.save_pretrained(ONNX_INT8_DIR)




In [ ]:
print_model_size(ONNX_INT8_DIR, "ONNX INT8")

In [ ]:
trainer.model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)
trainer.model.to(trainer.args.device)

pt_val_output = trainer.predict(tokenized_val)
print("PyTorch val:", pt_val_output.metrics)

In [ ]:
from optimum.onnxruntime import pipeline as ort_pipeline
from optimum.onnxruntime import ORTModelForSequenceClassification
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(ONNX_INT8_DIR)
int8_model = ORTModelForSequenceClassification.from_pretrained(
    ONNX_INT8_DIR,
    file_name="model_quantized.onnx",
)

classifier = ort_pipeline("text-classification", model=int8_model, tokenizer=tokenizer,device=-1,)

preds = classifier(list(macd_val["text"]), batch_size=64, truncation=True)
pred_ids = [LABEL2ID[p["label"]] for p in preds]
labels = list(macd_val["label"])

int8_acc = accuracy_score(labels, pred_ids)
int8_f1 = f1_score(labels, pred_ids, average="macro")
print("INT8 val:", {"accuracy": int8_acc, "f1_macro": int8_f1})
print("Δ accuracy:", int8_acc - 0.8448275862068966)
print("Δ f1_macro:", int8_f1 - 0.8446819785126103)